In [ ]:
###------ TF-IDF + Model Zoo ------###

# Level 1 — Screening Model (TF-IDF + Model Zoo)

Goal: Train a Baseline Binary Classifier (relevant=1, irrelevant=0) for **title+abstract screening**  and Generate:
- Probability of **How relevant is the paper**
- Ranking 
- Decision **INCLUDE / MAYBE / EXCLUDE** with two-threshold policy

--------------------------------------------------------------------------------------------------------------------------------
We are gonna use 2 **Dummy Datasets**
- screening_dataset.csv  (labeled training data)
- new_records.csv        (unlabeled incoming records)

 **Import Libraries**

In [ ]:
#libraries:

import numpy as np
import pandas as pd
from pathlib import Path
import joblib

from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import recall_score, precision_score, f1_score, average_precision_score

## 2. Helper functions (text cleaning + threshold policy)

Define basic Helper Functions to keep the pipeline clea:

### (a) `normalize()`
 basic cleaning in strings 

### (b) `merge_text(title, abstract)`
Φτιάχνει το τελικό input του μοντέλου:
- `text = title + " " + abstract`
- αν λείπει abstract, χρησιμοποιεί μόνο title.

**το μοντέλο βλέπει ενιαίο κείμενο αντί για ξεχωριστά fields**

### (c) `find_t_include(...)`
choose the threshold point in order to achieve:
**Recall ≥ target_recall** in validation set.

## How it works:
Σκανάρει thresholds από υψηλό προς χαμηλό:
- το πρώτο threshold που πιάνει τον στόχο recall → ελαχιστοποιεί το workload (burden).

### (d) `DECISION_MAP`
Υλοποιεί το **two-threshold decision**:
- `p ≥ T_include` → INCLUDE
- `p ≤ T_exclude` → EXCLUDE
- αλλιώς → MAYBE (region of uncertainty)

In [ ]:
# -----------------------------
# Helpers
# -----------------------------
def normalize(s: str) -> str:
    s = "" if pd.isna(s) else str(s)
    return " ".join(s.split())

def merge_text(title, abstract) -> str:
    t = normalize(title)
    a = normalize(abstract)
    return (t + " " + a).strip() if a else t.strip()

def find_t_include(y_true, proba_pos, target_recall=0.95, step=0.01) -> float:
    for t in np.arange(1.0, -step, -step):
        y_pred = (proba_pos >= t).astype(int)
        if recall_score(y_true, y_pred, zero_division=0) >= target_recall:
            return float(t)
    return 0.0

DECISION_MAP = {2: "INCLUDE", 1: "MAYBE", 0: "EXCLUDE"}


In [ ]:
# -----------------------------
# Paths (SAME FOLDER)
# -----------------------------
BASE_DIR = Path.cwd()
screening_path = BASE_DIR / "screening_dataset.csv"
new_path = BASE_DIR / "new_records.csv"

print("Screening exists:", screening_path.exists())
print("New records exists:", new_path.exists())


In [ ]:
# -----------------------------
# Load & prepare data
# -----------------------------
df = pd.read_csv(screening_path)
df["text"] = [merge_text(t, a) for t, a in zip(df["title"], df["abstract"])]
df["label"] = df["label"].astype(int)

X = df["text"].tolist()
y = df["label"].to_numpy()

X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.25, random_state=42, stratify=y
)


## 5. Baseline model: TF-IDF + Logistic Regression    Baseline model Zoo (1 model)

Train the baseline suite model

### TF-IDF vectorizer
- `ngram_range=(1,2)`: unigrams + bigrams (useful for phrases such as “hazard ratio”).
-  `max_features=50000`: limits the vocabulary.
- `stop_words="english"`: removes common words which don't help
- `strip_accents="unicode"`: clean characters

### Logistic Regression
- `class_weight="balanced"`: useful for class imbalance
- `solver="liblinear"`:constant for binary classification.
- `max_iter=2000`: so that it converges

## Model Zoo (4 Models)

| ID | Model                             |
| -- | --------------------------------- |
| M1 | TF-IDF + Logistic Regression      |
| M2 | TF-IDF + Linear SVM (calibrated)  |
| M3 | TF-IDF + Multinomial Naive Bayes  |
| M4 | TF-IDF + SGDClassifier (log-loss) |

- TDF Input
- Generate probabilities (or calibrated scores)

**Design Rule:**

- Same vectorizer (same params)
- Same train/val split
- Same target_recall
- Same T_exclude


In [ ]:
#libraries for model Zoo:

from sklearn.svm import LinearSVC
from sklearn.calibration import CalibratedClassifierCV
from sklearn.naive_bayes import MultinomialNB
from sklearn.linear_model import SGDClassifier

**Model Zoo Function**

In [ ]:
def make_tfidf():
    return TfidfVectorizer(
        ngram_range=(1, 2),
        max_features=50000,
        stop_words="english",
        lowercase=True,
        strip_accents="unicode",
    )

def build_model_zoo(random_state=42):
    zoo = {
        "tfidf_logreg": Pipeline([
            ("tfidf", make_tfidf()),
            ("clf", LogisticRegression(
                max_iter=2000,
                class_weight="balanced",
                solver="liblinear",
                random_state=random_state
            ))
        ]),

        "tfidf_svm": Pipeline([
            ("tfidf", make_tfidf()),
            ("clf", CalibratedClassifierCV(
                LinearSVC(class_weight="balanced", random_state=random_state),
                method="sigmoid",
                cv=5
            ))
        ]),

        "tfidf_nb": Pipeline([
            ("tfidf", make_tfidf()),
            ("clf", MultinomialNB())
        ]),

        "tfidf_sgd": Pipeline([
            ("tfidf", make_tfidf()),
            ("clf", SGDClassifier(
                loss="log_loss",
                class_weight="balanced",
                max_iter=2000,
                random_state=random_state
            ))
        ]),
    }
    return zoo


## 6. Thresholding for screening + validation metrics

Goal is **not to lose relevant** so **AUC** is not the right metric to use

### 2- threshold Policy
- Put `target_recall = 0.95`
- choose `T_include` that Recall ≥ 0.95 in validation.
- Put `T_exclude = 0.10` (relevant low) for “safe excludes”.

### Metrics που αναφέρουμε
- **Recall@Include** (primary): how many relevant "catch".
- **Precision@Include**: workload proxy (how clean is the include set).
- **F1@Include**: summary.
- **PR-AUC**: useful for imbalance.
- **Burden**: porpotion of records is INCLUDE ( workload metric).

> Output: prints the validation metrics and the thresholds (`T_include`, `T_exclude`).


In [ ]:
# -----------------------------
# Thresholding (screening logic)
# -----------------------------
target_recall = 0.95
t_include = find_t_include(y_val, val_proba, target_recall)
t_exclude = 0.10

y_pred = (val_proba >= t_include).astype(int)

metrics = {
    "Recall@Include": recall_score(y_val, y_pred, zero_division=0),
    "Precision@Include": precision_score(y_val, y_pred, zero_division=0),
    "F1@Include": f1_score(y_val, y_pred, zero_division=0),
    "PR-AUC": average_precision_score(y_val, val_proba),
    "Burden": (val_proba >= t_include).mean(),
    "T_include": t_include,
    "T_exclude": t_exclude
}

print("\nVALIDATION METRICS")
for k, v in metrics.items():
    print(f"{k}: {v}")

**Unified evaluation function (screening-aware)**

In [ ]:
def evaluate_model(
    model_name,
    pipeline,
    X_train, y_train,
    X_val, y_val,
    target_recall=0.95,
    t_exclude=0.10
):
    pipeline.fit(X_train, y_train)

    if hasattr(pipeline, "predict_proba"):
        val_proba = pipeline.predict_proba(X_val)[:, 1]
    else:
        raise ValueError(f"{model_name} does not support probability output")

    t_include = find_t_include(
        y_val, val_proba, target_recall=target_recall
    )

    y_pred = (val_proba >= t_include).astype(int)
    n_include = int((val_proba >= t_include).sum())
    n_total = len(val_proba)

    return {
        "model": model_name,
        "target_recall": target_recall,
        "t_include": t_include,
        "t_exclude": t_exclude,
        "recall": recall_score(y_val, y_pred, zero_division=0),
        "precision": precision_score(y_val, y_pred, zero_division=0),
        "f1": f1_score(y_val, y_pred, zero_division=0),
        "pr_auc": average_precision_score(y_val, val_proba),
        "burden": float((val_proba >= t_include).mean()),
        "n_include": n_include,
        "burden": n_include / n_total
    }


**Run Model Zoo and its Evaluation**

In [ ]:
model_zoo = build_model_zoo(random_state=42)

results = []
for name, pipe in model_zoo.items():
    print(f"Evaluating {name}...")
    res = evaluate_model(
        name, pipe,
        X_train, y_train,
        X_val, y_val,
        target_recall=0.95,
        t_exclude=0.10
    )
    results.append(res)

results_df = pd.DataFrame(results).sort_values("burden")
results_df


**Pick best model + train on full labeled + score new records**

In [ ]:
#Choose the best model

def pick_best_model(results_df, target_recall=0.95):
    df_ok = results_df[results_df["recall"] >= target_recall].copy()
    if df_ok.empty:
        raise ValueError("No model achieved target recall. Lower target_recall or check data/model setup.")

    df_ok = df_ok.sort_values(
        by=["burden", "pr_auc", "precision", "t_include"],
        ascending=[True, False, False, False]
    ).reset_index(drop=True)

    return df_ok.iloc[0], df_ok

best_row, ranked_models = pick_best_model(results_df, target_recall=0.95)
best_row, ranked_models


In [ ]:
#Train the chosen model on all labeled data (train+val)

best_model_name = best_row["model"]
best_t_include = float(best_row["t_include"])
best_t_exclude = float(best_row["t_exclude"])

best_pipe = model_zoo[best_model_name]
best_pipe.fit(X, y)  # train on full labeled dataset

print("SELECTED MODEL:", best_model_name)
print("T_include:", best_t_include, "T_exclude:", best_t_exclude)


In [ ]:
#Score + rank new records with the selected model

new_df = pd.read_csv(new_path)
new_df["text"] = [merge_text(t, a) for t, a in zip(new_df["title"], new_df["abstract"])]

proba = best_pipe.predict_proba(new_df["text"].tolist())[:, 1]
new_df["proba_relevant"] = proba

decision_code = np.ones(len(new_df), dtype=int)
decision_code[proba >= best_t_include] = 2
decision_code[proba <= best_t_exclude] = 0
new_df["decision"] = [DECISION_MAP[c] for c in decision_code]

new_df = new_df.sort_values("proba_relevant", ascending=False).reset_index(drop=True)
new_df.insert(0, "rank", new_df.index + 1)

new_df[["rank", "record_id", "title", "proba_relevant", "decision"]].head(20)


## 7. Inference on new records (ranking + INCLUDE/MAYBE/EXCLUDE)

screening workflow:

1. upload `new_records.csv`.
2. create `text = title + abstract`.
3. compute `proba_relevant` for each record.
4. apply the  two-threshold policy:
   - INCLUDE: high probability
   - EXCLUDE: low probability
   - MAYBE: uncertainty → human review
5.  sorting according to `proba_relevant` (descending) and make `rank`.

> Output: print the papers sorted top ranking. This is the list for Reviewer

In [ ]:
# -----------------------------
# Export Reviewer Pack (EXCEL)
# -----------------------------
from pathlib import Path

export_cols = [
    "rank",
    "record_id",
    "title",
    "proba_relevant",
    "decision",
]

# Optional metadata if available
for c in ["year", "journal", "authors", "doi", "pmid", "url"]:
    if c in new_df.columns and c not in export_cols:
        export_cols.append(c)

reviewer_df = new_df.copy()

# Add human review columns
reviewer_df["human_label"] = ""        # INCLUDE / EXCLUDE
reviewer_df["human_reason"] = ""       # short rationale
reviewer_df["needs_fulltext"] = ""     # YES / NO
reviewer_df["notes"] = ""              # optional comments

# Keep abstract if exists (important for screening)
if "abstract" in reviewer_df.columns:
    export_cols_final = export_cols + ["abstract"]
else:
    export_cols_final = export_cols

export_cols_final += [
    "human_label",
    "human_reason",
    "needs_fulltext",
    "notes"
]

out_file = Path("reviewer_pack.xlsx")

with pd.ExcelWriter(out_file, engine="openpyxl") as writer:
    reviewer_df[export_cols_final].to_excel(
        writer,
        index=False,
        sheet_name="Screening"
    )

print("✅ Reviewer Pack written to:", out_file.resolve())
